In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import tensorflow_probability.substrates.jax as tfp

from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic, shapelets
from gigalens.jax.profiles.mass import epl, shear
# NOTE: the direct old-API imports (PhysicalModel, LensSimulator,
# Forward/BackwardProbModel) were removed — they were only used by the
# commented-out manual-construction cell below and are being retired in the
# scene-API migration. The active path builds everything via the
# gigalens_research pipeline (get_inference_builder -> ProbModel).

import jax
from jax import random
import numpy as np
import json
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
import blackjax
import importlib
import os
tfd = tfp.distributions

In [ ]:
from gigalens_research.inference import MCLMC
from gigalens_research.inference_utils import *
from gigalens_research.plotting import *
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.scene_prob_model import Dataset, ProbModel
# from gigalens_research.simtests.registry import get_inference_builder

In [ ]:
home = os.path.expanduser("~/")
srcdir = os.path.join(home, "gigalens/src/")
data_dir = os.path.join(home, f"GIGALens-Code/data/")
results_dir = os.path.join(home, f"GIGALens-Code/results/testsys60/")


In [ ]:
f = np.load(os.path.join(data_dir, "simulated_systems/100SystemsStandard80px.npz"))
keys = f.files
observed_imgs = jnp.array([f[key] for key in keys])


def params_lists_to_jax(params_list):
    """Convert nested parameter structure of Python lists back to JAX arrays"""
    params = []
    for i in range(len(params_list)):
        params.append([])
        for j in range(len(params_list[i])):
            params[i].append({})
            for key in params_list[i][j]:
                params[i][j][key] = jnp.array(params_list[i][j][key])
    return params
    # return jax.tree.map(lambda a : jnp.array(a), params_list, is_leaf=lambda x : isinstance(x, list) and isinstance(x[0], int))

filename = os.path.join(data_dir, 'simulated_systems/100SystemsStandardParams.yaml')
with open(filename, 'r') as file:
    true_params = params_lists_to_jax(yaml.safe_load(file))

i = 60
# observed_img = observed_imgs[i]

new_obs = np.load(os.path.join(home, "GIGALens-Code/results/why_hard_to_sample_worktree/resim/sys60_ss16", "observed_ss16.npz"))
observed_img = new_obs['m8'] + new_obs['residual_noise']
# plt.imshow(new_obs['m128'])

# select_index = lambda a : a[i]
# sys_60_true = jax.tree.map(select_index, true_params)

In [ ]:
epl0 = Component(epl.EPL(50), dict(
    theta_E=tfd.LogNormal(jnp.log(1.25), 0.4),
    gamma=tfd.TruncatedNormal(2, 0.5, 1, 3),
    e1=tfd.Normal(0, 0.2),
    e2=tfd.Normal(0, 0.2),
    center_x=tfd.Normal(0, 0.06),
    center_y=tfd.Normal(0, 0.06),
))

shear0 = Component(shear.Shear(), dict(gamma1=tfd.Normal(0, 0.1), gamma2=tfd.Normal(0, 0.1)))

lens_light = Component(sersic.SersicEllipse(use_lstsq=False), dict(
    R_sersic=tfd.LogNormal(jnp.log(1.6), 0.25),
    n_sersic=tfd.Uniform(0.5, 8),
    e1=tfd.TruncatedNormal(0, 0.1, -0.15, 0.15),
    e2=tfd.TruncatedNormal(0, 0.1, -0.15, 0.15),
    center_x=tfd.Normal(0, 0.02),
    center_y=tfd.Normal(0, 0.02),
    Ie=tfd.LogNormal(jnp.log(300.0), 0.5),
))

source_light = Component(sersic.SersicEllipse(use_lstsq=False), dict(
    R_sersic=tfd.LogNormal(jnp.log(0.25), 0.25),
    n_sersic=tfd.Uniform(0.5, 8),
    e1=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
    e2=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
    center_x=tfd.Normal(0, 0.5),
    center_y=tfd.Normal(0, 0.5),
    Ie=tfd.LogNormal(jnp.log(150.0), 0.9),
))

model = LensModel([
    Plane(mass=[epl0, shear0], light=[lens_light]),
    Plane(light=[source_light])
])

kernel = np.load(os.path.join(srcdir, 'gigalens/assets/psf.npy')).astype(np.float64)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=80, supersample=2, kernel=kernel)

ds = Dataset(observed_img, sim_config, sees=[lens_light, source_light], background_rms=0.2, exp_time=100)

prob_model = ProbModel(model, ds, mode="forward")

In [ ]:
pipeline = Pipeline(InferenceContext.from_prob_model(prob_model))


#! Start from true params
# start_z = jnp.array(prob_model.bij.inverse(sys_60_true))

# pipeline_config = PipelineConfig(steps=["SVI", "HMC"],
#                                     svi_kwargs=dict(start=start_z, num_steps=5000, n_vi=1000),
#                                     hmc_kwargs=dict(num_burnin_steps=500, num_results=8000, n_hmc=64))
pipeline.add(MAPStage(num_steps=350, n_samples=200))
pipeline.add(SVIStage(num_steps=1000, n_vi=500))
pipeline.add(MCLMCStage(n_chains=8, num_burnin_steps=2000, num_results=2000, progress_bar=True, debug=True))

In [ ]:
artifacts = pipeline.run(resume=True, out_dir=results_dir)

In [ ]:
ps = pipeline.posterior()
report = PosteriorReport(ps) #, truth_x=true_params, truth_source_fn=truth_fn
pipeline_report = PipelineReport(pipeline)

In [ ]:
report.source_panel()
plt.show()

In [ ]:

fig = pipeline_report.diagnostics("mclmc", chain=3)
fig.show()

In [ ]:
report.full_report()
plt.show()

In [ ]:

# fig = pipeline_report.compound_corner()

In [ ]:
fig = pipeline_report.image_comparison()

In [ ]:
fig = pipeline_report.loss_histories()
